In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import re

In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\deppa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\deppa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\deppa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# Load CSV
df = pd.read_csv("text.csv")

# Preprocess text
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply preprocessing
df['text'] = df['text'].apply(preprocess_text)

# Remove stopwords
stop_words = set(stopwords.words('english'))
df['text'] = df['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

# Tokenize
df['text'] = df['text'].apply(word_tokenize)

# Stemming
stemmer = PorterStemmer()
df['text'] = df['text'].apply(lambda x: [stemmer.stem(word) for word in x])
df['text'] = df['text'].apply(lambda x: ' '.join(x))

In [4]:
df

,Unnamed: 0,text,label
0,0,feel realli helpless heavi heart,4
1,1,ive enjoy abl slouch relax unwind frankli need...,0
2,2,gave internship dmrg feel distraught,4
3,3,dont know feel lost,0
4,4,kindergarten teacher thoroughli weari job take...,4
...,...,...,...
416804,416804,feel like tell horni devil find site suit sort...,2
416805,416805,began realiz feel agit restless would thought ...,3
416806,416806,feel curiou previou earli dawn time seek troubl,5
416807,416807,feel becuas tyran natur govern el salvador sav...,3


In [6]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)

# Vectorize text
vectorizer = TfidfVectorizer(max_features=1000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train Naive Bayes model
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# Test accuracy
y_pred = model.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 81.12%


In [1]:
# Save this as app.py
import streamlit as st

# Title
st.title("AI-Powered Text Emotion Analyzer")

# Upload CSV file
st.subheader("Upload CSV File (Optional)")
uploaded_file = st.file_uploader("Upload CSV with 'text' column", type=["csv"])

# Input text box
st.subheader("Enter Text")
user_input = st.text_area("Type your text here:", height=100)

# Button to analyze
if st.button("Analyze Emotion"):
    if uploaded_file is not None:
        df = pd.read_csv(uploaded_file)
        texts = df['text'].tolist()
    elif user_input:
        texts = [user_input]
    else:
        st.error("Please enter text or upload a CSV file.")
        st.stop()

    # Preprocess and predict
    results = []
    for text in texts:
        # Preprocess
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        text = ' '.join([word for word in text.split() if word not in stop_words])
        text = ' '.join([stemmer.stem(word) for word in text.split()])
        
        # Vectorize
        text_vec = vectorizer.transform([text])
        
        # Predict
        emotion = model.predict(text_vec)
        confidence = model.predict_proba(text_vec).max()
        
        # Suggestion
        suggestions = {
            "sadness": "Try to rephrase: 'I feel a bit down today' instead of 'I am so sad'.",
            "anger": "Try to rephrase: 'I am frustrated' instead of 'I hate this'.",
            "love": "Keep it! It's a positive emotion.",
            "surprise": "Try to rephrase: 'I am shocked' instead of 'I am so surprised'.",
            "fear": "Try to rephrase: 'I am anxious' instead of 'I am scared'.",
            "joy": "Keep it! It's a positive emotion."
        }
        suggestion = suggestions.get(emotion, "No suggestion available.")

        results.append({
            "text": text,
            "emotion": emotion,
            "confidence": f"{confidence * 100:.2f}%",
            "suggestion": suggestion
        })

    # Display results
    for result in results:
        st.write(f"**Text:** {result['text']}")
        st.write(f"**Emotion:** {result['emotion'].title()}")
        st.write(f"**Confidence:** {result['confidence']}")
        st.write(f"**Suggestion:** {result['suggestion']}")
        st.markdown("---")

2026-03-02 17:12:58.652 
  command:

    streamlit run C:\Users\deppa\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-02 17:12:58.654 Session state does not function when running a script without `streamlit run`
